# P2.3R Replication, P2.3R-bis Certification, P2.3T Threshold

This notebook loads the recorded artifacts of the replication line without needing a model endpoint. Aggregates are regenerated from per-run episode logs so metric definitions stay consistent.


In [ ]:
import json
from pathlib import Path

from paradigm.p23r import aggregate_p23r

summary = aggregate_p23r(Path('../results/core_p23r'))
for c in summary['cells']:
    print(c['model'], c['arrival'], 'TTR', c['ttr_episodes']['median'], 'deploy', c['time_to_deployment_episode']['median'], 'exposure', c['post_promotion_novel_exposure']['median'])


Certification boundary per teacher: shadow acceptance grouped by novel episodes in the candidate's train split.


In [ ]:
for b in summary['certification_boundary']:
    for row in b['by_novel_train_episodes']:
        print(b['model'], row['novel_train_episodes'], row['candidates'], row['promoted'], round(row['acceptance_min'], 2), round(row['acceptance_max'], 2))


P2.3R-bis pairs and the negative control, and the P2.3T learning curve.


In [ ]:
bis = json.loads(Path('../results/core_p23r_bis/core_p23r_bis_summary.json').read_text())
for row in bis['pairs']:
    nc = row.get('negative_control') or {}
    print(row['arrival'], 'delay', row.get('certification_delay_episodes'), 'poisoned:', nc.get('poisoned_candidate', {}).get('recent', {}).get('outcome'), '/', nc.get('poisoned_candidate', {}).get('family_aware', {}).get('outcome'))

threshold = json.loads(Path('../results/core_p23t/core_p23t_threshold.json').read_text())
for model, res in threshold.items():
    for c in res['curve']:
        print(model, c['novel_train_episodes'], round(c['capability_accuracy_mean'], 2), round(c['gate_acceptance_mean'], 2), c['recent_promote_rate'], c['family_aware_promote_rate'])


P2.4 Type B: capability versus evidence, and the frozen post-stream evaluation.


In [ ]:
p24 = json.loads(Path('../results/core_p24/core_p24_type_b.json').read_text())
for model, block in p24['models'].items():
    print(model, block['offline_class'], 'TTC', block['sweep']['time_to_capability']['median'])
    for c in block['sweep']['curve']:
        print('  k', c['novel_train_episodes'], 'acc', round(c['decision_accuracy_mean'], 2), 'replay', round(c['replay_success_mean'], 2), 'gate', round(c['gate_acceptance_mean'], 2))
    for mode, r in (block.get('online') or {}).items():
        ps = r['post_stream_exposure']
        print('  online', mode, 'hybrid', ps['success_rate'], 'llm-only', ps['baseline_success_rate'], 'reflex-only', ps['reflex_only_success_rate'], 'zero-LLM solves', ps['solved_with_zero_llm_calls'])
